# 📁 Veri Setleri — Google Drive

Bu notebook'ta kullanılan **ham ve temizlenmiş veri setleri** aşağıdaki Google Drive klasöründe bulunmaktadır:

🔗 **[Veri Setlerine Erişmek İçin Tıklayın](https://drive.google.com/drive/u/0/folders/1rEm79GRlGlZZit9HWRcPRZKSVoq6TQS0)**

> Klasörde şu dosyalar yer almaktadır:
> - `rayli_sistem_detay.csv` — Ham veriden derlenen, henüz temizlenmemiş raylı sistem yolcu verisi
> - `rayli_sistem_temiz.csv` — Durak adı normalizasyonu ve gruplama işlemlerinden geçirilmiş temiz veri

---

# Data Cleaning and Aggregation
## İstanbul Raylı Sistem Yolcu Verisi

Bu notebook, İstanbul Metro ham yolcu verisinin temizleme ve birleştirme sürecini belgelemektedir.

**Problem:** Ham veride aynı fiziksel istasyon, farklı giriş/çıkış kapılarına (KUZEY/GUNEY, BATI/DOGU) ve numaralı varyantlara (-1, -2, -3) göre ayrı satırlar olarak kayıtlı. Bu durum;
- Yanlış durak bazlı yolcu sayılarına,
- Aynı `(hat, durak, tarih, saat)` için mükerrer satırlara yol açmaktadır.

**Çözüm:** İki aşamalı temizlik — durak adı normalizasyonu + gruplama ile toplama.

---
**Pipeline Özeti:**
```
hourly_transportation_*.csv  →  [analizer.py]  →  rayli_sistem_detay.csv  →  [durak_birlestir.py]  →  rayli_sistem_temiz.csv
```

---
## 0. Gerekli Kütüphaneler

In [ ]:
import pandas as pd
import re
import duckdb
import os
import glob
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

print('Kütüphaneler yüklendi ✓')

---
## 1. Ham Verinin Toplanması — `analizer.py`

Ham veri, aylık `hourly_transportation_*.csv` dosyalarından DuckDB ile çekilir. Sadece `road_type = 'RAYLI'` satırları alınır; `34A` hattı hariç tutulur. Tüm dosyalar tek sorguda birleştirilerek `rayli_sistem_detay.csv` oluşturulur.

In [ ]:
# --- Dosya yolları ---
masaustu   = os.path.join(os.path.expanduser('~'), 'Desktop')
klasor     = os.path.join(masaustu, 'proje_optimization')
dosya_kalip = os.path.join(klasor, 'hourly_transportation_*.csv').replace('\\', '/')
hedef_dosya = os.path.join(klasor, 'rayli_sistem_detay.csv')

bulunan = glob.glob(os.path.join(klasor, 'hourly_transportation_*.csv'))
print(f'Klasör        : {klasor}')
print(f'Bulunan dosya : {len(bulunan)} adet')

if len(bulunan) == 0:
    raise FileNotFoundError('Kaynak CSV dosyaları bulunamadı!')

In [ ]:
# --- DuckDB sorgusu: tüm dosyaları tek sorguda birleştir ---
query = f"""
SELECT 
    TRIM(line_name)           AS hat_adi,
    TRIM(station_poi_desc_cd) AS durak_adi,
    transition_date           AS tarih,
    transition_hour           AS saat,
    SUM(number_of_passage)    AS toplam_yolcu
FROM '{dosya_kalip}'
WHERE road_type = 'RAYLI'
  AND TRIM(line_name) != '34A'
GROUP BY 1, 2, 3, 4
ORDER BY hat_adi, durak_adi, tarih, saat
"""

print('DuckDB sorgusu çalıştırılıyor... (bu adım 2-5 dk sürebilir)')
df_raw = duckdb.query(query).df()

# Tip düzenlemeleri
df_raw['hat_adi']        = df_raw['hat_adi'].str.strip()
df_raw['durak_adi']      = df_raw['durak_adi'].str.strip()
df_raw['saat']           = df_raw['saat'].apply(lambda s: f'{int(s):02d}')
df_raw['toplam_yolcu']   = df_raw['toplam_yolcu'].astype(int)
df_raw = df_raw[['hat_adi', 'durak_adi', 'tarih', 'saat', 'toplam_yolcu']]

# Kaydet
df_raw.to_csv(hedef_dosya, index=False, encoding='utf-8-sig')
print(f'Toplam {len(df_raw):,} satır → {hedef_dosya}')
df_raw.head(5)

---
## 2. Ham Verinin Yüklenmesi ve Keşifsel Analiz

In [ ]:
INPUT_PATH  = r'C:\Users\Lenovo\Desktop\proje_optimization\rayli_sistem_detay.csv'
OUTPUT_PATH = r'C:\Users\Lenovo\Desktop\proje_optimization\rayli_sistem_temiz.csv'

df = pd.read_csv(INPUT_PATH, dtype=str)
df['toplam_yolcu'] = pd.to_numeric(df['toplam_yolcu'], errors='coerce').fillna(0).astype(int)

print('=== Ham Veri Özeti ===')
print(f'Toplam satır          : {len(df):,}')
print(f'Benzersiz hat         : {df["hat_adi"].nunique()}')
print(f'Benzersiz durak adı   : {df["durak_adi"].nunique()}')
print(f'Tarih aralığı         : {df["tarih"].min()}  →  {df["tarih"].max()}')
print()
df.head(5)

In [ ]:
# Mükerrer satır tespiti: aynı (hat, durak, tarih, saat) > 1 kez mi?
dupcheck = df.groupby(['hat_adi', 'durak_adi', 'tarih', 'saat']).size().reset_index(name='kayit_sayisi')
dups     = dupcheck[dupcheck['kayit_sayisi'] > 1]

print(f'Mükerrer (hat+durak+tarih+saat) grubu sayısı : {len(dups):,}')
print(f'Toplam etkilenen satır                       : {dups["kayit_sayisi"].sum() - len(dups):,} fazladan satır')
print()
print('Örnek mükerrer gruplar:')
dups.sort_values('kayit_sayisi', ascending=False).head(8)

In [ ]:
# Sorunlu durak adlarının dağılımı
PROBLEM_KEYWORDS = ['KUZEY', 'GUNEY', 'BATI', 'DOGU', 'GIRISI', 'HOL', 'TERMINAL', 'KONKORS']

all_stations = sorted(df['durak_adi'].dropna().unique())
problem_stations = [s for s in all_stations
                    if any(k in str(s).upper() for k in PROBLEM_KEYWORDS)
                    or re.search(r'[-\s]+\d+$', str(s))]

print(f'Toplam benzersiz durak          : {len(all_stations)}')
print(f'Normalizasyon gerektiren durak  : {len(problem_stations)}')
print()
print('İlk 20 sorunlu durak adı:')
for s in problem_stations[:20]:
    print(f'  {s}')

---
## 3. Durak Adı Normalizasyonu — `durak_birlestir.py`

İki katmanlı strateji uygulanır:

| Katman | Yöntem | Kullanım Amacı |
|--------|--------|----------------|
| **Manuel Harita** | `MANUAL_MAP` sözlüğü | Karmaşık veya istisna vakalar (LEVENT ailesi, MAHMUTBEY HOL'ları, M9 çift adları…) |
| **Regex Temizleme** | `SUFFIX_PATTERNS` listesi | Genel suffix'ler: KUZEY/GUNEY, BATI/DOGU, -1/-2/-3, (BATI), (DOGU/ADLIYE)… |

In [ ]:
# ── Manuel Eşleştirme Tablosu ────────────────────────────────────────────────
MANUAL_MAP = {
    'ITU GUNEY': 'ITU', 'ITU KUZEY': 'ITU',
    'LEVENT 2 KUZEY': 'LEVENT', 'LEVENT GUNEY': 'LEVENT',
    'LEVENT BATI KONKORS': 'LEVENT', 'LEVENT DOGU KONKORS': 'LEVENT',
    '4 LEVENT 2 GUNEY': '4. LEVENT', '4 LEVENT KUZEY': '4. LEVENT',
    'MAHMUTBEY M3 HOL 3': 'MAHMUTBEY', 'MAHMUTBEY M3 HOL 4': 'MAHMUTBEY',
    'MAHMUTBEY M7 HOL 1': 'MAHMUTBEY', 'MAHMUTBEY M7 HOL 2': 'MAHMUTBEY',
    'MAHMUTBEY M7 HOL 3': 'MAHMUTBEY',
    'M7 YILDIZ 1': 'YILDIZ', 'M7 YILDIZ 2': 'YILDIZ',
    'M9 - 29 EKIM CUMHURIYET - IHLAS YUVA': '29 EKIM CUMHURIYET',
    'M9 29 EKIM CUMHURIYET-IHLAS YUVA': '29 EKIM CUMHURIYET',
    'M9 15 TEMMUZ': '15 TEMMUZ',
    'M9 HALKALI GUNEY': 'HALKALI', 'M9 HALKALI KUZEY': 'HALKALI',
    'HAVALIMANI  (TERMINAL 1)': 'HAVALIMANI', 'HAVALIMANI (TERMINAL 1)': 'HAVALIMANI',
    'SIRKECI-1': 'SIRKECI', 'SIRKECI-2': 'SIRKECI', 'SIRKECI-3': 'SIRKECI',
    'TERSANE-1': 'TERSANE', 'TERSANE-2': 'TERSANE',
    'USKUDAR 1': 'USKUDAR', 'USKUDAR 2': 'USKUDAR',
    'USKUDAR-1': 'USKUDAR', 'USKUDAR-2': 'USKUDAR', 'USKUDAR-3': 'USKUDAR',
    'YENIKAPI-1': 'YENIKAPI', 'YENIKAPI-2': 'YENIKAPI', 'YENIKAPI-3': 'YENIKAPI',
    'YENIKAPI GUNEY': 'YENIKAPI', 'YENIKAPI KUZEY': 'YENIKAPI',
    'BAKIRKOY-1': 'BAKIRKOY', 'BAKIRKOY-2': 'BAKIRKOY',
    'BOSTANCI-1': 'BOSTANCI', 'BOSTANCI-2': 'BOSTANCI',
    'GEBZE-1': 'GEBZE', 'GEBZE-2': 'GEBZE',
    'KUCUKYALI-1': 'KUCUKYALI', 'KUCUKYALI-2': 'KUCUKYALI',
    'CEKMEKOY 1': 'CEKMEKOY', 'CEKMEKOY 2': 'CEKMEKOY',
    'ALTUNIZADE 1': 'ALTUNIZADE', 'ALTUNIZADE 2': 'ALTUNIZADE',
    'AKSARAY 1': 'AKSARAY', 'OTOGAR 1': 'OTOGAR',
    'EMINONU 2': 'EMINONU', 'ZEYTINBURNU 2': 'ZEYTINBURNU', 'KABATAS 2': 'KABATAS',
    'SEYRANTEPE 1 BATI': 'SEYRANTEPE', 'SEYRANTEPE 2 DOGU': 'SEYRANTEPE',
    'SEYRANTEPE 3 STAD GIRISI': 'SEYRANTEPE',
    'OZGURLUK MEYDANI GUNEY': 'OZGURLUK MEYDANI',
}

# ── Regex Suffix Desenleri ───────────────────────────────────────────────────
SUFFIX_PATTERNS = [
    r'\s*(KUZEY|GUNEY|GÜNEY|BATI|DOGU|DOĞU)$',        # yön suffix'i
    r'\s*\((BATI|DOGU|DOĞU|KUZEY|GUNEY|GÜNEY)[^)]*\)$', # parantezli yön
    r'\s*\(ADLIYE\)$',                                  # özel: DOGU/ADLIYE
    r'[-\s]+\d+$',                                      # -1, -2, 1, 2 sonu
    r'^\d+\s+',                                         # 1 , 2 başı
]

def normalize(name: str) -> str:
    name = str(name).strip().upper()
    if name in MANUAL_MAP:
        return MANUAL_MAP[name]
    cleaned = name
    for pat in SUFFIX_PATTERNS:
        cleaned = re.sub(pat, '', cleaned).strip()
    return cleaned if cleaned else name

print('Normalizasyon fonksiyonları tanımlandı ✓')

# Fonksiyonu birkaç örnek üzerinde test et
test_cases = [
    ('ITU GUNEY', 'ITU'),
    ('4 LEVENT 2 GUNEY', '4. LEVENT'),
    ('MAHMUTBEY M7 HOL 2', 'MAHMUTBEY'),
    ('ACIBADEM (BATI)', 'ACIBADEM'),
    ('SIRKECI-3', 'SIRKECI'),
    ('YENIKAPI-2', 'YENIKAPI'),
    ('M9 HALKALI KUZEY', 'HALKALI'),
]

print(f"\n{'GİRİŞ':<40} {'BEKLENEN':<25} {'SONUÇ':<25} {'OK?'}")
print('-' * 100)
for raw, expected in test_cases:
    result = normalize(raw)
    ok = '✓' if result == expected else '✗'
    print(f'{raw:<40} {expected:<25} {result:<25} {ok}')

In [ ]:
# Tüm veriye normalizasyonu uygula
df['durak_adi_temiz'] = df['durak_adi'].apply(normalize)

# Değişen satırları raporla
mapping_report = (
    df[df['durak_adi'] != df['durak_adi_temiz']]
    [['durak_adi', 'durak_adi_temiz']]
    .drop_duplicates()
    .sort_values('durak_adi')
    .reset_index(drop=True)
)

print(f'Normalize edilen benzersiz durak adı : {len(mapping_report)}')
print()
mapping_report.columns = ['Eski Ad', 'Yeni Ad']
mapping_report

---
## 4. Gruplama ve Yolcu Sayısı Toplama

Normalizasyon sonrası aynı `(hat_adi, durak_adi_temiz, tarih, saat)` grubuna düşen satırların `toplam_yolcu` değerleri toplanır.

In [ ]:
df_clean = (
    df
    .groupby(['hat_adi', 'durak_adi_temiz', 'tarih', 'saat'], as_index=False)
    ['toplam_yolcu'].sum()
    .rename(columns={'durak_adi_temiz': 'durak_adi'})
    [['hat_adi', 'durak_adi', 'tarih', 'saat', 'toplam_yolcu']]
)

print('=== Temizlik Sonuçları ===')
print(f'Satır sayısı (önce)   : {len(df):>10,}')
print(f'Satır sayısı (sonra)  : {len(df_clean):>10,}')
print(f'Azaltılan satır       : {len(df)-len(df_clean):>10,}  ({(len(df)-len(df_clean))/len(df)*100:.1f}%)')
print(f'Durak sayısı (önce)   : {df["durak_adi"].nunique():>10}')
print(f'Durak sayısı (sonra)  : {df_clean["durak_adi"].nunique():>10}')
print()
df_clean.head(5)

---
## 5. Doğrulama — Temizlik Öncesi vs Sonrası

Toplam yolcu sayısı korunmalı; sadece satır ve durak sayısı azalmalıdır.

In [ ]:
toplam_once  = df['toplam_yolcu'].sum()
toplam_sonra = df_clean['toplam_yolcu'].sum()
fark         = toplam_once - toplam_sonra

print('=== Yolcu Sayısı Doğrulaması ===')
print(f'Toplam yolcu (önce)   : {toplam_once:>15,}')
print(f'Toplam yolcu (sonra)  : {toplam_sonra:>15,}')
print(f'Fark                  : {fark:>15,}')
print()
if fark == 0:
    print('✅ Yolcu sayısı korundu — veri kaybı yok!')
else:
    print(f'⚠️  Dikkat: {fark:,} yolcu farkı var, incelenmeli!')

In [ ]:
# En çok birleştirilen duraklar — hangi istasyon en fazla çoğaltılmıştı?
merge_stats = (
    df.groupby('durak_adi_temiz')['durak_adi']
    .nunique()
    .reset_index()
    .rename(columns={'durak_adi': 'varyant_sayisi', 'durak_adi_temiz': 'durak_adi'})
    .sort_values('varyant_sayisi', ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(merge_stats['durak_adi'][::-1], merge_stats['varyant_sayisi'][::-1],
               color='steelblue', edgecolor='white')
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_xlabel('Birleştirilen Varyant Sayısı')
ax.set_title('En Fazla Varyantı Olan İstasyonlar (Top 15)', fontsize=13, fontweight='bold')
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

In [ ]:
# Hat bazında satır azalma oranı
hat_once  = df.groupby('hat_adi').size().rename('once')
hat_sonra = df_clean.groupby('hat_adi').size().rename('sonra')
hat_karsi = pd.concat([hat_once, hat_sonra], axis=1).fillna(0).astype(int)
hat_karsi['azalma_pct'] = ((hat_karsi['once'] - hat_karsi['sonra']) / hat_karsi['once'] * 100).round(1)
hat_karsi = hat_karsi.sort_values('azalma_pct', ascending=False)

print('Hat bazında satır azalma oranları:')
hat_karsi

---
## 6. Temizlenmiş Verinin Kaydedilmesi

In [ ]:
df_clean.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f'✅ Temizlenmiş veri kaydedildi:')
print(f'   {OUTPUT_PATH}')
print(f'   {len(df_clean):,} satır  |  {df_clean["durak_adi"].nunique()} benzersiz durak  |  {df_clean["hat_adi"].nunique()} hat')

---
## 7. Özet

| Metrik | Ham Veri | Temiz Veri |
|--------|----------|------------|
| Toplam satır | 4,222,304 | 3,455,244 |
| Benzersiz durak | 348 | 266 |
| Azaltılan satır | — | 767,060 (%18.2) |
| Toplam yolcu | ✓ korundu | ✓ korundu |

**Temizlik Adımları:**
1. `analizer.py` — DuckDB ile ham aylık dosyaları birleştirip `rayli_sistem_detay.csv` oluşturur
2. `durak_birlestir.py` — Manuel harita + regex ile durak adlarını normalize eder, gruplama ile yolcu sayılarını toplar

Veri, analize hazır hale getirilmiştir.